# Analisis Harga Rumah Jakarta Selatan

Notebook ini berisi: pembersihan data, statistika deskriptif, uji normalitas, korelasi, regresi linear berganda (OLS), dan uji beda harga rumah dengan/tanpa garasi.

**Cara pakai:** jalankan sel dari atas ke bawah (Runtime → Run all). Saat diminta, unggah file Excel `3TIF_06_HargaRumahJaksel.xlsx` (sheet bernama `Data`).

## 0. Import library & pengaturan

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

ALPHA = 0.05
OUT = "output"
os.makedirs(OUT, exist_ok=True)
sns.set_theme(style="whitegrid")

# Samakan tampilan di layar Colab dengan gambar yang disimpan ke laporan
# (Colab memakai 72 dpi secara bawaan, sedangkan file PNG disimpan 200 dpi)
plt.rcParams["figure.dpi"] = 200
plt.rcParams["savefig.dpi"] = 200

log_lines = []

def log(*args):
    text = " ".join(str(a) for a in args)
    print(text)
    log_lines.append(text)

## 1. Unggah & bersihkan data

In [ ]:
# Unggah file Excel (jika file sudah ada di folder Colab, unggahan dilewati)
from google.colab import files

NAMA_FILE = "3TIF_06_HargaRumahJaksel.xlsx"
if os.path.exists(NAMA_FILE):
    path = NAMA_FILE
else:
    uploaded = files.upload()
    path = list(uploaded.keys())[0]
print("File yang dipakai:", path)

In [ ]:
df = pd.read_excel(path, sheet_name="Data")
df.columns = [str(c).strip().upper() for c in df.columns]
log("Kolom terbaca:", list(df.columns))
log("Jumlah baris awal:", len(df))

# Kolom wajib. Sesuaikan nama jika di file Excel berbeda.
WAJIB = ["HARGA", "LT", "LB", "GRS"]
hilang = [c for c in WAJIB if c not in df.columns]
if hilang:
    raise ValueError(f"Kolom {hilang} tidak ditemukan. Sesuaikan nama kolom di sel ini.")

for c in ["HARGA", "LT", "LB", "JKT", "JKM"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
df["GRS"] = df["GRS"].astype(str).str.strip().str.upper()

log("\n== Missing value per kolom ==")
log(df.isna().sum().to_string())

n_dup = df.duplicated().sum()
log("\nBaris duplikat:", n_dup)
df = df.drop_duplicates().reset_index(drop=True)
df = df.dropna(subset=["HARGA", "LT", "LB", "GRS"]).reset_index(drop=True)
log("Jumlah baris setelah pembersihan:", len(df))

df["LN_HARGA"] = np.log(df["HARGA"])
df["GARASI"] = np.where(df["GRS"] == "ADA", "Ada garasi", "Tanpa garasi")
df.head()

## 2. Statistika deskriptif

In [ ]:
kolom_num = [c for c in ["HARGA", "LT", "LB", "JKT", "JKM"] if c in df.columns]
desk = df[kolom_num].agg(
    ["count", "mean", "median", "std", "min", "max"]
).T
desk["Q1"] = df[kolom_num].quantile(0.25)
desk["Q3"] = df[kolom_num].quantile(0.75)
desk["skewness"] = df[kolom_num].skew()
desk["kurtosis"] = df[kolom_num].kurt()  # excess kurtosis
desk.to_csv(f"{OUT}/tabel_deskriptif.csv")
log("\n== Statistika deskriptif ==")
log(desk.round(3).to_string())

log("\n== Frekuensi garasi ==")
log(df["GARASI"].value_counts().to_string())

In [ ]:
# Histogram + boxplot
for c, label in [("HARGA", "Harga (Rp)"), ("LT", "Luas Tanah (m²)"),
                 ("LB", "Luas Bangunan (m²)"), ("LN_HARGA", "ln(Harga)")]:
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    # Jumlah bin dibuat tetap agar hasil sama di semua versi numpy/Colab:
    # aturan Freedman-Diaconis (fd), dibatasi maksimal 62 bin
    n_bin = min(len(np.histogram_bin_edges(df[c], bins="fd")) - 1, 62)
    sns.histplot(df[c], bins=n_bin, kde=True, ax=ax[0], color="#2F5597")
    ax[0].set_xlabel(label)
    ax[0].set_ylabel("Frekuensi")
    sns.boxplot(x=df[c], ax=ax[1], color="#9DC3E6")
    ax[1].set_xlabel(label)
    plt.tight_layout()
    plt.savefig(f"{OUT}/gambar_hist_box_{c.lower()}.png", dpi=200)
    plt.show()

## 3. Uji normalitas

In [ ]:
log("\n== Uji normalitas (H0: data berdistribusi normal) ==")
hasil_norm = []
for c in ["HARGA", "LN_HARGA", "LT", "LB"]:
    x = df[c]
    sw_w, sw_p = stats.shapiro(x)
    k2, dp_p = stats.normaltest(x)
    ks_stat, ks_p = stats.kstest((x - x.mean()) / x.std(ddof=1), "norm")
    hasil_norm.append({
        "Variabel": c, "Shapiro_W": sw_w, "Shapiro_p": sw_p,
        "DAgostino_K2": k2, "DAgostino_p": dp_p,
        "KS_D": ks_stat, "KS_p": ks_p,
        "Skewness": x.skew(), "Kurtosis": x.kurt(),
    })
norm_df = pd.DataFrame(hasil_norm)
norm_df.to_csv(f"{OUT}/tabel_normalitas.csv", index=False)
log(norm_df.round(4).to_string(index=False))
log("Catatan: p < 0,05 -> tolak H0 (tidak normal).")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
stats.probplot(df["HARGA"], dist="norm", plot=ax[0])
ax[0].set_title("Q-Q Plot Harga")
stats.probplot(df["LN_HARGA"], dist="norm", plot=ax[1])
ax[1].set_title("Q-Q Plot ln(Harga)")
plt.tight_layout()
plt.savefig(f"{OUT}/gambar_qq_harga.png", dpi=200)
plt.show()

## 4. Korelasi

In [ ]:
log("\n== Korelasi ==")
var_kor = ["HARGA", "LN_HARGA", "LT", "LB"]
pearson = df[var_kor].corr(method="pearson")
spearman = df[var_kor].corr(method="spearman")
pearson.to_csv(f"{OUT}/tabel_korelasi_pearson.csv")
spearman.to_csv(f"{OUT}/tabel_korelasi_spearman.csv")
log("Pearson:\n" + pearson.round(4).to_string())
log("Spearman:\n" + spearman.round(4).to_string())

log("\nUji signifikansi korelasi terhadap HARGA dan LN_HARGA:")
rows = []
for y in ["HARGA", "LN_HARGA"]:
    for x in ["LT", "LB"]:
        r, p = stats.pearsonr(df[x], df[y])
        rho, p_s = stats.spearmanr(df[x], df[y])
        rows.append({"Y": y, "X": x, "Pearson_r": r, "p_Pearson": p,
                     "Spearman_rho": rho, "p_Spearman": p_s})
kor_df = pd.DataFrame(rows)
kor_df.to_csv(f"{OUT}/tabel_uji_korelasi.csv", index=False)
log(kor_df.round(4).to_string(index=False))

In [ ]:
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(spearman.loc[["HARGA", "LT", "LB"], ["HARGA", "LT", "LB"]],
            annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1)
plt.title("Matriks Korelasi Spearman")
plt.tight_layout()
plt.savefig(f"{OUT}/gambar_heatmap_korelasi.png", dpi=200)
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.scatterplot(data=df, x="LT", y="HARGA", ax=ax[0], alpha=0.5)
ax[0].set_xlabel("Luas Tanah (m²)")
ax[0].set_ylabel("Harga (Rp)")
sns.scatterplot(data=df, x="LB", y="HARGA", ax=ax[1], alpha=0.5)
ax[1].set_xlabel("Luas Bangunan (m²)")
ax[1].set_ylabel("Harga (Rp)")
plt.tight_layout()
plt.savefig(f"{OUT}/gambar_scatter.png", dpi=200)
plt.show()

## 5. Regresi linear berganda (OLS)

In [ ]:
log("\n== Regresi linear berganda ==")


def fit_ols(y_col):
    X = sm.add_constant(df[["LT", "LB"]])
    model = sm.OLS(df[y_col], X).fit()
    return X, model


for y_col in ["HARGA", "LN_HARGA"]:
    X, model = fit_ols(y_col)
    log(f"\n--- Model: {y_col} ~ LT + LB ---")
    log(model.summary().as_text())
    vif = pd.DataFrame({
        "Variabel": X.columns[1:],
        "VIF": [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])],
    })
    log("VIF:\n" + vif.round(4).to_string(index=False))
    bp_lm, bp_p, _, _ = het_breuschpagan(model.resid, X)
    log(f"Breusch-Pagan: LM = {bp_lm:.4f}, p = {bp_p:.4g}")
    sw_w, sw_p = stats.shapiro(model.resid)
    log(f"Shapiro-Wilk residual: W = {sw_w:.4f}, p = {sw_p:.4g}")
    # Robust standard error (HC3) sebagai pembanding bila heteroskedastis
    rob = model.get_robustcov_results(cov_type="HC3")
    log("Koefisien dengan SE robust (HC3):\n" + str(rob.summary().tables[1]))

    coef = pd.DataFrame({
        "Koefisien": model.params, "Std_Error": model.bse,
        "t": model.tvalues, "p_value": model.pvalues,
    })
    coef.to_csv(f"{OUT}/tabel_regresi_{y_col.lower()}.csv")

    plt.figure(figsize=(5.5, 4.2))
    plt.scatter(model.fittedvalues, model.resid, alpha=0.5)
    plt.axhline(0, color="red", lw=1)
    plt.xlabel("Nilai prediksi")
    plt.ylabel("Residual")
    plt.title(f"Residual vs Prediksi ({y_col})")
    plt.tight_layout()
    plt.savefig(f"{OUT}/gambar_residual_{y_col.lower()}.png", dpi=200)
    plt.show()

## 6. Uji beda harga: ada garasi vs tanpa garasi

In [ ]:
log("\n== Uji beda harga: ada garasi vs tanpa garasi ==")
a = df.loc[df["GARASI"] == "Ada garasi", "HARGA"]
b = df.loc[df["GARASI"] == "Tanpa garasi", "HARGA"]
la = df.loc[df["GARASI"] == "Ada garasi", "LN_HARGA"]
lb = df.loc[df["GARASI"] == "Tanpa garasi", "LN_HARGA"]
log(f"n ada garasi = {len(a)}, n tanpa garasi = {len(b)}")
log(f"Rata-rata harga: ada = {a.mean():,.0f} ; tanpa = {b.mean():,.0f}")
log(f"Median harga   : ada = {a.median():,.0f} ; tanpa = {b.median():,.0f}")

lev_w, lev_p = stats.levene(la, lb, center="median")
log(f"Levene (ln harga): W = {lev_w:.4f}, p = {lev_p:.4g}")
t_stat, t_p = stats.ttest_ind(la, lb, equal_var=False)  # Welch
log(f"Welch t-test (ln harga): t = {t_stat:.4f}, p = {t_p:.4g}")
u_stat, u_p = stats.mannwhitneyu(a, b, alternative="two-sided")
log(f"Mann-Whitney U (harga): U = {u_stat:.1f}, p = {u_p:.4g}")

# Ukuran efek (Cohen d pada ln harga)
sp = np.sqrt(((len(la) - 1) * la.var(ddof=1) + (len(lb) - 1) * lb.var(ddof=1))
             / (len(la) + len(lb) - 2))
log(f"Cohen's d (ln harga): {(la.mean() - lb.mean()) / sp:.4f}")

plt.figure(figsize=(5.5, 4.2))
sns.boxplot(data=df, x="GARASI", y="HARGA", hue="GARASI",
            palette=["#9DC3E6", "#F4B183"], legend=False)
plt.xlabel("Ketersediaan garasi")
plt.ylabel("Harga (Rp)")
plt.tight_layout()
plt.savefig(f"{OUT}/gambar_box_garasi.png", dpi=200)
plt.show()

## 7. Simpan & unduh hasil

In [ ]:
with open(f"{OUT}/hasil_analisis.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(log_lines))
print(f"Selesai. Hasil tersimpan di folder '{OUT}/'.")

# Kompres folder output lalu unduh
import shutil
shutil.make_archive("output", "zip", OUT)
files.download("output.zip")